In [1]:
import pandas as pd

df = pd.read_csv("Dataset/t20i_Matches_Data.csv")

C:\Users\arman\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = df[df["Match Format"] == "T20"].copy()

In [3]:
df = df[df["Match Winner"].notna()].copy()

In [4]:
df = df[df["Match Winner"].notna()].copy()

In [5]:
keep_cols = [
    "Match Date",
    "Team1 Name",
    "Team2 Name",
    "Team1 Runs Scored",
    "Team2 Runs Scored",
    "Team1 Wickets Fell",
    "Team2 Wickets Fell",
    "Match Venue (Country)",
    "Match Venue (City)",
    "Toss Winner",
    "Toss Winner Choice",
    "Match Winner"
]

df = df[keep_cols]

In [6]:
df["Match Date"] = pd.to_datetime(df["Match Date"])
df = df.sort_values("Match Date").reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2494 entries, 0 to 2493
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Match Date             2484 non-null   datetime64[ns]
 1   Team1 Name             2494 non-null   object        
 2   Team2 Name             2494 non-null   object        
 3   Team1 Runs Scored      2493 non-null   float64       
 4   Team2 Runs Scored      2492 non-null   float64       
 5   Team1 Wickets Fell     2493 non-null   float64       
 6   Team2 Wickets Fell     2492 non-null   float64       
 7   Match Venue (Country)  2494 non-null   object        
 8   Match Venue (City)     2494 non-null   object        
 9   Toss Winner            2493 non-null   object        
 10  Toss Winner Choice     2494 non-null   object        
 11  Match Winner           2494 non-null   object        
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 233.

In [7]:
df = df.dropna(subset=[
    "Match Date",
    "Team1 Runs Scored",
    "Team2 Runs Scored",
    "Team1 Wickets Fell",
    "Team2 Wickets Fell",
    "Toss Winner"
]).reset_index(drop=True)

In [8]:
df["team1_win"] = (df["Match Winner"] == df["Team1 Name"]).astype(int)
df["toss_advantage"] = (df["Toss Winner"] == df["Team1 Name"]).astype(int)

In [9]:
team_rows = []

for _, r in df.iterrows():
    team_rows.append({
        "Match Date": r["Match Date"],
        "Team": r["Team1 Name"],
        "Runs Scored": r["Team1 Runs Scored"],
        "Runs Conceded": r["Team2 Runs Scored"],
        "Win": int(r["Match Winner"] == r["Team1 Name"])
    })
    team_rows.append({
        "Match Date": r["Match Date"],
        "Team": r["Team2 Name"],
        "Runs Scored": r["Team2 Runs Scored"],
        "Runs Conceded": r["Team1 Runs Scored"],
        "Win": int(r["Match Winner"] == r["Team2 Name"])
    })

team_df = pd.DataFrame(team_rows).sort_values("Match Date").reset_index(drop=True)
team_df.head()

,Match Date,Team,Runs Scored,Runs Conceded,Win
0,2006-06-15,Sri Lanka,163.0,161.0,1
1,2006-06-15,England,161.0,163.0,0
2,2006-08-28,England,144.0,148.0,0
3,2006-08-28,Pakistan,148.0,144.0,1
4,2006-11-28,Bangladesh,166.0,123.0,1


In [10]:
window = 10

team_df["win_pct_last_10"] = (
    team_df.groupby("Team")["Win"]
    .rolling(window, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

team_df["avg_runs_scored_last_10"] = (
    team_df.groupby("Team")["Runs Scored"]
    .rolling(window, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

team_df["avg_runs_conceded_last_10"] = (
    team_df.groupby("Team")["Runs Conceded"]
    .rolling(window, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

team_df.head(12)

,Match Date,Team,Runs Scored,Runs Conceded,Win,win_pct_last_10,avg_runs_scored_last_10,avg_runs_conceded_last_10
0,2006-06-15,Sri Lanka,163.0,161.0,1,1.000000,163.000000,161.000000
1,2006-06-15,England,161.0,163.0,0,0.000000,161.000000,163.000000
2,2006-08-28,England,144.0,148.0,0,0.000000,152.500000,155.500000
3,2006-08-28,Pakistan,148.0,144.0,1,1.000000,148.000000,144.000000
4,2006-11-28,Bangladesh,166.0,123.0,1,1.000000,166.000000,123.000000
5,2006-11-28,Zimbabwe,123.0,166.0,0,0.000000,123.000000,166.000000
6,2006-12-01,South Africa,126.0,127.0,0,0.000000,126.000000,127.000000
7,2006-12-01,India,127.0,126.0,1,1.000000,127.000000,126.000000
8,2006-12-22,Sri Lanka,62.0,162.0,1,1.000000,112.500000,161.500000
9,2006-12-22,New Zealand,162.0,62.0,0,0.000000,162.000000,62.000000


In [11]:
# Team1 features
df = df.merge(
    team_df[[
        "Match Date", "Team",
        "win_pct_last_10",
        "avg_runs_scored_last_10",
        "avg_runs_conceded_last_10"
    ]],
    left_on=["Match Date", "Team1 Name"],
    right_on=["Match Date", "Team"],
    how="left"
)

df = df.rename(columns={
    "win_pct_last_10": "team1_win_pct",
    "avg_runs_scored_last_10": "team1_avg_runs_scored",
    "avg_runs_conceded_last_10": "team1_avg_runs_conceded"
}).drop(columns=["Team"])

In [12]:
# Team2 features
df = df.merge(
    team_df[[
        "Match Date", "Team",
        "win_pct_last_10",
        "avg_runs_scored_last_10",
        "avg_runs_conceded_last_10"
    ]],
    left_on=["Match Date", "Team2 Name"],
    right_on=["Match Date", "Team"],
    how="left"
)

df = df.rename(columns={
    "win_pct_last_10": "team2_win_pct",
    "avg_runs_scored_last_10": "team2_avg_runs_scored",
    "avg_runs_conceded_last_10": "team2_avg_runs_conceded"
}).drop(columns=["Team"])

In [13]:
df.head()

,Match Date,Team1 Name,Team2 Name,Team1 Runs Scored,Team2 Runs Scored,Team1 Wickets Fell,Team2 Wickets Fell,Match Venue (Country),Match Venue (City),Toss Winner,Toss Winner Choice,Match Winner,team1_win,toss_advantage,team1_win_pct,team1_avg_runs_scored,team1_avg_runs_conceded,team2_win_pct,team2_avg_runs_scored,team2_avg_runs_conceded
0,2006-06-15,Sri Lanka,England,163.0,161.0,10.0,5.0,England,Southampton,Sri Lanka,bat,Sri Lanka,1,1,1.0,163.0,161.0,0.0,161.0,163.0
1,2006-08-28,England,Pakistan,144.0,148.0,7.0,5.0,England,Bristol,England,bat,Pakistan,0,1,0.0,152.5,155.5,1.0,148.0,144.0
2,2006-11-28,Bangladesh,Zimbabwe,166.0,123.0,10.0,9.0,Bangladesh,Khulna,Zimbabwe,bowl,Bangladesh,1,0,1.0,166.0,123.0,0.0,123.0,166.0
3,2006-12-01,South Africa,India,126.0,127.0,9.0,4.0,South Africa,Johannesburg,South Africa,bat,India,0,1,0.0,126.0,127.0,1.0,127.0,126.0
4,2006-12-22,New Zealand,Sri Lanka,162.0,62.0,8.0,1.0,New Zealand,Wellington,New Zealand,bat,Sri Lanka,0,1,0.0,162.0,62.0,1.0,112.5,161.5


In [14]:
df.isna().sum()

Match Date                 0
Team1 Name                 0
Team2 Name                 0
Team1 Runs Scored          0
Team2 Runs Scored          0
Team1 Wickets Fell         0
Team2 Wickets Fell         0
Match Venue (Country)      0
Match Venue (City)         0
Toss Winner                0
Toss Winner Choice         0
Match Winner               0
team1_win                  0
toss_advantage             0
team1_win_pct              0
team1_avg_runs_scored      0
team1_avg_runs_conceded    0
team2_win_pct              0
team2_avg_runs_scored      0
team2_avg_runs_conceded    0
dtype: int64

In [15]:
df["win_pct_diff"] = df["team1_win_pct"] - df["team2_win_pct"]
df["avg_runs_scored_diff"] = df["team1_avg_runs_scored"] - df["team2_avg_runs_scored"]
df["avg_runs_conceded_diff"] = df["team1_avg_runs_conceded"] - df["team2_avg_runs_conceded"]

In [16]:
features = [
    "win_pct_diff",
    "avg_runs_scored_diff",
    "avg_runs_conceded_diff",
    "toss_advantage"
]

X = df[features]
y = df["team1_win"]

In [17]:
train = df["Match Date"] < "2020-01-01"

X_train, X_test = X[train], X[~train]
y_train, y_test = y[train], y[~train]

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Log Loss:", log_loss(y_test, y_prob))

Accuracy: 0.7387344199424737
Log Loss: 0.5135134272355772


In [23]:
import pickle

with open("t20_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [19]:
!pip install xgboost

In [20]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Log Loss:", log_loss(y_test, y_prob))

Accuracy: 0.7334611697027804
Log Loss: 0.5661648804715029


In [21]:
import pickle

with open("t20_xgb_model.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

In [24]:
team_df.to_csv("team_stats.csv", index=False)